In [1]:
from model import *

ModuleNotFoundError: No module named 'model'

In [ ]:
import optax
from optax.losses import softmax_cross_entropy_with_integer_labels

from Addition_Transformer.model import forward
def loss_fn(weights: jax.Array, token_ids: jax.Array, mask: jax.Array) -> jax.Array:
    logits = forward(token_ids[:, :-1], weights)
    targets = token_ids[:, 1:]
    # for addition, we need to build a mask because the model needs to only predict the last 3 digits
    # need mask code
    loss_mask = mask[:, 1:]
    loss = softmax_cross_entropy_with_integer_labels(logits, targets) # mean would also change since masking the outputs
    loss = jnp.sum(loss * loss_mask) / jnp.sum(loss_mask)
    return loss

In [ ]:
total_steps = cfg.num_epochs * cfg.batch_size
schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=cfg.lr,          # e.g. 3e-4
    warmup_steps=total_steps * 0.05,   # ~5-10% of total steps
    decay_steps=total_steps,     # total training steps, NOT epochs
    end_value=cfg.lr * 0.1,     # floor, often peak/10
)
optimizer = optax.adamw(learning_rate=schedule, weight_decay=0.1)

In [ ]:
def calc_val_loss(val_loader, weights):
    val_losses = []
    for x, mask in val_loader:
        loss = loss_fn(weights, x, mask)
        val_losses.append(loss.item())
    return np.mean(val_losses)
@jax.jit
def train_step(x, mask, weights, opt_state):
    loss, grads = jax.value_and_grad(loss_fn)(weights, x, mask)
    updates, opt_state = optimizer.update(grads, opt_state, weights)
    weights = optax.apply_updates(weights, updates)
    return loss, weights, opt_state

In [ ]:
from tqdm import tqdm
def train(train_dataset, val_dataset, train_masks, val_masks, weights, cfg: Config):
    train_loader = Dataloader(train_dataset, train_masks, cfg.batch_size)
    val_loader = Dataloader(val_dataset, val_masks, cfg.batch_size)
    opt_state = optimizer.init(weights)
    train_losses = []
    val_losses = []
    for epoch in range(cfg.num_epochs):
        local_losses = []
        for batch_idx, (x, mask) in enumerate(tqdm(train_loader)):
            loss, weights, opt_state = train_step(x, mask, weights, opt_state) # moving to function for jax jit
            local_losses.append(loss.item())

        avg_train_loss = np.mean(local_losses)
        val_loss = calc_val_loss(val_loader, weights)
        print(f"Epoch {epoch+1}/{cfg.num_epochs}, Train Loss: {avg_train_loss}, Val Loss: {val_loss}")
        train_losses.append(avg_train_loss)
        val_losses.append(val_loss)

        # TODO: add training checkpoint

    return weights, train_losses, val_losses



In [ ]:
weights = Weights.init(cfg, jax.random.key(0))
weights, train_losses, val_losses = train(train_dataset, val_dataset, train_masks, val_masks, weights, cfg)

In [ ]:
# plot training and validation losses
import matplotlib.pyplot as plt
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.legend()
plt.show()